**Steps**

1. Load the fixed text reference created in Notebook 01.
2. Download only the reports belonging to the selected 2,200 studies.
3. Read the radiology report files.
4. Extract the main clinical sections.
5. Clean the text.
6. Check missing and empty reports.
7. Save the processed text data.

**Output:** `text_processed.csv`

In [28]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import re
import subprocess
from tqdm.auto import tqdm

base_path = '/content/drive/MyDrive/dissertation_project/data'

raw_path = f'{base_path}/raw'
processed_path = f'{base_path}/processed'

report_download_path = f'{raw_path}/selected_reports'

os.makedirs(report_download_path, exist_ok=True)
os.makedirs(processed_path, exist_ok=True)

print("Report folder:", report_download_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Report folder: /content/drive/MyDrive/dissertation_project/data/raw/selected_reports


In [29]:
import getpass
physionet_user = input("PhysioNet username/email: ")
physionet_pass = getpass.getpass("PhysioNet password: ")

PhysioNet username/email: BhavishyaGuntreddi
PhysioNet password: ··········


In [30]:
# Load the text reference
text_reference = pd.read_csv(
    f'{processed_path}/3_text_reference.csv'
)

print(
    "Text reference shape:",
    text_reference.shape
)

print("\nColumns:")
print(
    text_reference.columns.tolist()
)

text_reference.head()

Text reference shape: (2200, 3)

Columns:
['subject_id', 'study_id', 'report_path']


,subject_id,study_id,report_path
0,10003052,58630288,files/p10/p10003052/s58630288.txt
1,10011126,58239923,files/p10/p10011126/s58239923.txt
2,10015701,53321493,files/p10/p10015701/s53321493.txt
3,10015931,57792054,files/p10/p10015931/s57792054.txt
4,10018081,57486705,files/p10/p10018081/s57486705.txt


In [31]:
# Check the report paths
print(
    text_reference[
        ['study_id', 'report_path']
    ].head(10)
)

   study_id                        report_path
0  58630288  files/p10/p10003052/s58630288.txt
1  58239923  files/p10/p10011126/s58239923.txt
2  53321493  files/p10/p10015701/s53321493.txt
3  57792054  files/p10/p10015931/s57792054.txt
4  57486705  files/p10/p10018081/s57486705.txt
5  58430738  files/p10/p10018286/s58430738.txt
6  56501836  files/p10/p10037020/s56501836.txt
7  59082434  files/p10/p10058697/s59082434.txt
8  54217246  files/p10/p10068304/s54217246.txt
9  50650870  files/p10/p10070011/s50650870.txt


In [32]:
# PhysioNet URLs
base_url = (
    'https://physionet.org/files/'
    'mimic-cxr/2.1.0/'
)


text_reference['report_url'] = (
    base_url
    + text_reference['report_path']
      .astype(str)
      .str.lstrip('/')
)

print(
    text_reference[
        ['study_id', 'report_path', 'report_url']
    ].head()
)

   study_id                        report_path  \
0  58630288  files/p10/p10003052/s58630288.txt   
1  58239923  files/p10/p10011126/s58239923.txt   
2  53321493  files/p10/p10015701/s53321493.txt   
3  57792054  files/p10/p10015931/s57792054.txt   
4  57486705  files/p10/p10018081/s57486705.txt   

                                          report_url  
0  https://physionet.org/files/mimic-cxr/2.1.0/fi...  
1  https://physionet.org/files/mimic-cxr/2.1.0/fi...  
2  https://physionet.org/files/mimic-cxr/2.1.0/fi...  
3  https://physionet.org/files/mimic-cxr/2.1.0/fi...  
4  https://physionet.org/files/mimic-cxr/2.1.0/fi...  


In [33]:
# Create local report paths
text_reference['local_report_path'] = (
    text_reference['study_id']
    .astype(str)
    .apply(
        lambda x: os.path.join(
            report_download_path,
            x + '.txt'
        )
    )
)

text_reference[
    ['study_id', 'local_report_path']
].head()

,study_id,local_report_path
0,58630288,/content/drive/MyDrive/dissertation_project/da...
1,58239923,/content/drive/MyDrive/dissertation_project/da...
2,53321493,/content/drive/MyDrive/dissertation_project/da...
3,57792054,/content/drive/MyDrive/dissertation_project/da...
4,57486705,/content/drive/MyDrive/dissertation_project/da...


In [34]:
# Test ONE report
test_row = text_reference.iloc[0]

test_output = test_row['local_report_path']

if os.path.exists(test_output):
    os.remove(test_output)

result = subprocess.run(
    [
        'wget',
        '-c',
        '--user=' + physionet_user,
        '--password=' + physionet_pass,
        '-O',
        test_output,
        test_row['report_url']
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

print("\nSTDERR:")
print(result.stderr)

if os.path.exists(test_output):
    print(
        "\nFile size:",
        os.path.getsize(test_output),
        "bytes"
    )

Return code: 0

STDERR:
--2026-08-24 12:53:24--  https://physionet.org/files/mimic-cxr/2.1.0/files/p10/p10003052/s58630288.txt
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized
Authentication selected: Basic realm="PhysioNet", charset="UTF-8"
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 414 [text/plain]
Saving to: ‘/content/drive/MyDrive/dissertation_project/data/raw/selected_reports/58630288.txt’

     0K                                                       100%  151M=0s

2026-08-24 12:53:25 (151 MB/s) - ‘/content/drive/MyDrive/dissertation_project/data/raw/selected_reports/58630288.txt’ saved [414/414]



File size: 414 bytes


In [35]:
# Read the test report
if os.path.exists(test_output):

    with open(
        test_output,
        'r',
        encoding='utf-8',
        errors='ignore'
    ) as f:


        test_report = f.read()
    print(test_report[:3000])

                                 FINAL REPORT
 INDICATION:  ___-year-old man with advanced dementia and 3 falls in the last 24
 hours.  Evaluate for acute pathology.
 
 TECHNIQUE:  Chest PA and lateral
 
 COMPARISON:  None.
 
 FINDINGS: 
 
 The lung volumes are low.  The lungs are clear without pleural effusion or
 pneumothorax.  The aorta is unfolded.  The heart size is normal.
 
 IMPRESSION: 
 
 Clear lungs.



In [36]:
# Download all selected reports
downloaded = 0
skipped = 0
failed = []

for _, row in tqdm(
    text_reference.iterrows(),
    total=len(text_reference),
    desc="Downloading radiology reports"
):

    output_path = row['local_report_path']

    # Skip only genuine non-empty files
    if (
        os.path.exists(output_path)
        and os.path.getsize(output_path) > 50
    ):
        skipped += 1
        continue

    # Remove old empty/invalid file
    if os.path.exists(output_path):
        os.remove(output_path)

    try:

        result = subprocess.run(
            [
                'wget',
                '-c',
                '--user=' + physionet_user,
                '--password=' + physionet_pass,
                '-O',
                output_path,
                row['report_url']
            ],
            capture_output=True,
            text=True
        )

        if (
            result.returncode == 0
            and os.path.exists(output_path)
            and os.path.getsize(output_path) > 50
        ):

            downloaded += 1

        else:

            failed.append({
                'study_id': row['study_id'],
                'status': result.returncode,
                'error': result.stderr[-500:]
            })

            if os.path.exists(output_path):
                os.remove(output_path)

    except Exception as e:

        failed.append({
            'study_id': row['study_id'],
            'status': 'ERROR',
            'error': str(e)
        })

        if os.path.exists(output_path):
            os.remove(output_path)

print("\nSuccessfully downloaded:", downloaded)
print("Already valid:", skipped)
print("Failed:", len(failed))


Successfully downloaded: 0
Already valid: 2200
Failed: 0


In [37]:
# Check downloaded reports
report_files = [
    f for f in os.listdir(report_download_path)
    if f.endswith('.txt')
]

valid_reports = [
    f for f in report_files
    if os.path.getsize(
        os.path.join(
            report_download_path,
            f
        )
    ) > 50
]

invalid_reports = [
    f for f in report_files
    if os.path.getsize(
        os.path.join(
            report_download_path,
            f
        )
    ) <= 50
]

print(
    "Total report files:",
    len(report_files)
)

print(
    "Valid reports:",
    len(valid_reports)
)

print(
    "Invalid/empty reports:",
    len(invalid_reports)
)

Total report files: 2200
Valid reports: 2200
Invalid/empty reports: 0


In [38]:
# Function to read reports
def read_report(path):

    try:

        with open(
            path,
            'r',
            encoding='utf-8',
            errors='ignore'
        ) as f:

            return f.read()

    except Exception:

        return ""

In [39]:
# Load the reports
text_reference['raw_report'] = (
    text_reference['local_report_path']
    .apply(read_report)
)

print(
    "Reports loaded:",
    len(text_reference)
)

print(
    "Empty reports:",
    (
        text_reference['raw_report']
        .str.strip()
        .eq('')
        .sum()
    )
)

Reports loaded: 2200
Empty reports: 0


In [40]:
# Extract FINDINGS and IMPRESSION
def extract_clinical_text(report):

    if not isinstance(report, str):
        return ""

    report = report.replace(
        '\r',
        '\n'
    )

    # Find the FINDINGS section
    findings_match = re.search(
        r'FINDINGS\s*:(.*?)(?=\n[A-Z][A-Z /_-]{2,30}\s*:|\Z)',
        report,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Find the IMPRESSION section
    impression_match = re.search(
        r'IMPRESSION\s*:(.*?)(?=\n[A-Z][A-Z /_-]{2,30}\s*:|\Z)',
        report,
        flags=re.IGNORECASE | re.DOTALL
    )

    findings = (
        findings_match.group(1)
        if findings_match
        else ""
    )

    impression = (
        impression_match.group(1)
        if impression_match
        else ""
    )

    # Combine the useful clinical sections
    combined = (
        "FINDINGS: "
        + findings.strip()
        + " "
        + "IMPRESSION: "
        + impression.strip()
    )

    return combined.strip()

In [41]:
# Apply the extraction
text_reference['clinical_text'] = (
    text_reference['raw_report']
    .apply(extract_clinical_text)
)

print(
    text_reference[
        [
            'study_id',
            'clinical_text'
        ]
    ].head(3)
)

   study_id                                      clinical_text
0  58630288  FINDINGS: The lung volumes are low.  The lungs...
1  58239923  FINDINGS: PA and lateral views of the chest.  ...
2  53321493  FINDINGS: Frontal and lateral views of the che...


In [42]:
# Text cleaning
def clean_text(text):

    if not isinstance(text, str):
        return ""

    # Replace line breaks with spaces
    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    # Remove repeated spaces
    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    # Remove leading/trailing spaces
    text = text.strip()

    return text

text_reference['report_text'] = (text_reference['clinical_text'].apply(clean_text))

In [43]:

# Check the processed text
print(
    text_reference[
        [
            'study_id',
            'report_text'
        ]
    ].head(5).to_string(
        index=False
    )
)

 study_id                                                                                                                                                                                                                                                                                                                                                                                           report_text
 58630288                                                                                                                                                                                                  FINDINGS: The lung volumes are low. The lungs are clear without pleural effusion or pneumothorax. The aorta is unfolded. The heart size is normal. IMPRESSION: Clear lungs. IMPRESSION: Clear lungs.
 58239923                                                                                                                                      FINDINGS: PA and lateral views of the chest. There is no 

In [44]:
# Check text length
text_reference['text_length'] = (
    text_reference['report_text']
    .str.len()
)


print(
    text_reference['text_length']
    .describe()
)

count    2200.000000
mean      459.689091
std       298.405768
min        21.000000
25%       279.750000
50%       395.500000
75%       580.000000
max      2544.000000
Name: text_length, dtype: float64


In [45]:
# Remove empty reports
before = len(text_reference)

text_reference = text_reference[
    text_reference['report_text']
    .str.strip()
    .ne('')
].copy()

after = len(text_reference)

print(
    "Before:",
    before
)

print(
    "After:",
    after
)

print(
    "Removed:",
    before - after
)

Before: 2200
After: 2200
Removed: 0


In [46]:
# Add the disease labels
structured_reference = pd.read_csv(
    f'{processed_path}/1_structured_reference.csv'
)

print(
    "Structured reference:",
    structured_reference.shape
)

print(
    structured_reference.columns.tolist()
)

Structured reference: (2200, 9)
['subject_id', 'study_id', 'No Finding', 'Support Devices', 'Pleural Effusion', 'Lung Opacity', 'Atelectasis', 'Cardiomegaly', 'Edema']


In [47]:
# Merge labels
label_columns = [
    'No Finding',
    'Support Devices',
    'Pleural Effusion',
    'Lung Opacity',
    'Atelectasis',
    'Cardiomegaly',
    'Edema'
]

available_labels = [
    col for col in label_columns
    if col in structured_reference.columns
]

print(
    "Available labels:",
    available_labels
)


text_final = text_reference.merge(
    structured_reference[
        ['study_id'] + available_labels
    ],
    on='study_id',
    how='left'
)

print(
    "Final text dataset:",
    text_final.shape
)

Available labels: ['No Finding', 'Support Devices', 'Pleural Effusion', 'Lung Opacity', 'Atelectasis', 'Cardiomegaly', 'Edema']
Final text dataset: (2200, 16)


In [48]:
# Keep the required columns
final_columns = [
    'subject_id',
    'study_id',
    'report_text'
] + available_labels

text_final = text_final[
    final_columns
].copy()

print(
    text_final.head()
)

   subject_id  study_id                                        report_text  \
0    10003052  58630288  FINDINGS: The lung volumes are low. The lungs ...   
1    10011126  58239923  FINDINGS: PA and lateral views of the chest. T...   
2    10015701  53321493  FINDINGS: Frontal and lateral views of the che...   
3    10015931  57792054  FINDINGS: IMPRESSION: Compared to preoperative...   
4    10018081  57486705  FINDINGS: As compared to the previous radiogra...   

   No Finding  Support Devices  Pleural Effusion  Lung Opacity  Atelectasis  \
0         1.0              NaN               NaN           NaN          NaN   
1         1.0              NaN               NaN           NaN          NaN   
2         1.0              NaN               NaN           NaN          NaN   
3         NaN              NaN               1.0           NaN          1.0   
4         NaN              1.0               1.0           NaN          1.0   

   Cardiomegaly  Edema  
0           NaN    NaN  
1     

In [49]:
# Final checks
print(
    "Final shape:",
    text_final.shape
)

print(
    "\nMissing text:",
    text_final['report_text']
    .isna()
    .sum()
)

print(
    "\nDuplicate studies:",
    text_final['study_id']
    .duplicated()
    .sum()
)

print(
    "\nUnique studies:",
    text_final['study_id']
    .nunique()
)

Final shape: (2200, 10)

Missing text: 0

Duplicate studies: 0

Unique studies: 2200


In [50]:
# Save the final output
output_file = (
    f'{processed_path}/text_processed.csv'
)

text_final.to_csv(
    output_file,
    index=False
)

print(
    "Saved:",
    output_file
)

print(
    "Shape:",
    text_final.shape
)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/text_processed.csv
Shape: (2200, 10)


In [51]:
print(text_final.shape)

(2200, 10)
